# Финансовый навигатор — решение (Logistic Regression, multi-label)

Мульти-лейбл классификация: вероятность интереса клиента к 9 продуктам.
Метрика платформы — **pooled/micro ROC-AUC** (один AUC по всей матрице клиент x продукт).

**Ключевой инсайт:** для pooled-метрики важна согласованность вероятностей *между*
лейблами. Линейная модель (LogReg + StandardScaler) даёт вероятности на едином
глобальном масштабе и на этой метрике обгоняет градиентный бустинг.

| модель | micro OOF |
|---|---|
| CatBoost | 0.659 |
| LightGBM | 0.664 |
| **LogReg (max_iter=1000)** | **0.670** |

Деревья и бленды с ними кросс-лейбл шкалу ухудшают, поэтому финальное решение —
чистый логрег. Образ: `flexonafft/ci-navigator:2.0`.

## 1. Импорты и константы

In [ ]:
import joblib, numpy as np, pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold

PRODUCTS = ["credit_card","mortgage","deposit","investment","insurance",
            "p2p_transfer","cashback","premium_account","business_loan"]
TARGETS = [f"product_{p}" for p in PRODUCTS]
FEATURES = ["age","income_bucket","tenure_months","tx_count_30d",
            "avg_tx_amount","digital_activity_score","has_child","is_salary_client"]
N_FOLDS, SEED, C, MAX_ITER = 5, 42, 1.0, 1000

## 2. Данные и EDA

In [ ]:
df = pd.read_csv("../train_weRmhWx.csv")
print("shape:", df.shape)
print(df[TARGETS].mean().round(3))

d = df.copy()
d["has_child"]=d["has_child"].astype(int); d["is_salary_client"]=d["is_salary_client"].astype(int)
corr = d[FEATURES+TARGETS].corr().loc[FEATURES,TARGETS].abs()
print("\nМакс |corr| признак->таргет:")
print(corr.max(axis=1).round(3))

Сигнал в `income_bucket`, `digital_activity_score`, `has_child`, `is_salary_client`;
остальные признаки близки к шуму. Зависимости преимущественно линейные/монотонные —
логрегу этого достаточно.

## 3. Признаки

In [ ]:
def make_features(frame):
    X = frame[FEATURES].copy()
    X["has_child"] = X["has_child"].astype(int)
    X["is_salary_client"] = X["is_salary_client"].astype(int)
    return X

X = make_features(df); Y = df[TARGETS].values

## 4. Честная pooled-OOF оценка (5-fold)

In [ ]:
skf = StratifiedKFold(N_FOLDS, shuffle=True, random_state=SEED)
oof = np.zeros((len(df), len(PRODUCTS)))
for j in range(len(PRODUCTS)):
    y = Y[:, j]
    for tr, va in skf.split(X, y):
        sc = StandardScaler().fit(X.iloc[tr])
        m = LogisticRegression(C=C, max_iter=MAX_ITER).fit(sc.transform(X.iloc[tr]), y[tr])
        oof[va, j] = m.predict_proba(sc.transform(X.iloc[va]))[:, 1]

macro = np.mean([roc_auc_score(Y[:,j], oof[:,j]) for j in range(len(PRODUCTS))])
micro = roc_auc_score(Y.ravel(), oof.ravel())
print(f"MACRO OOF AUC = {macro:.5f}")
print(f"MICRO OOF AUC = {micro:.5f}   <-- метрика платформы")

## 5. Финальная модель на всех данных + сохранение артефакта

In [ ]:
scaler = StandardScaler().fit(X)
Xs = scaler.transform(X)
models = [LogisticRegression(C=C, max_iter=MAX_ITER).fit(Xs, Y[:, j]) for j in range(len(PRODUCTS))]
joblib.dump({"scaler":scaler,"features":FEATURES,"products":PRODUCTS,"models":models}, "./models/model.joblib")
print("saved ./models/model.joblib")

## 6. Инференс (как в `run.py` внутри контейнера)

Контейнер получает `--input-path` / `--output-path`, пишет CSV без header,
9 колонок вероятностей в порядке `PRODUCTS`.

In [ ]:
def predict(input_path, output_path, models_dir="./models"):
    pack = joblib.load(f"{models_dir}/model.joblib")
    test = pd.read_csv(input_path)
    Xt = make_features(test)
    Xs = pack["scaler"].transform(Xt)
    preds = np.column_stack([m.predict_proba(Xs)[:,1] for m in pack["models"]])
    pd.DataFrame(preds).to_csv(output_path, header=False, index=False)
    return preds

df.head(200).to_csv("/tmp/_in.csv", index=False)
p = predict("/tmp/_in.csv", "/tmp/_out.csv")
print("output:", p.shape, "| диапазон:", round(p.min(),4), "..", round(p.max(),4))

## 7. Деплой

```bash
docker build --platform linux/amd64 -t flexonafft/ci-navigator:2.0 .
docker push flexonafft/ci-navigator:2.0
```
requirements: `scikit-learn==1.7.2`, `numpy==1.26.4`, `scipy==1.15.3`, `pandas==2.3.3`, `joblib==1.5.3`.